In [1]:
# call ltspice and simulate

import os
import subprocess
import numpy as np
from PyLTSpice import RawRead

def run_simulation(L, W, VG, Rd):
    # 1. SETTINGS
    LTSPICE_PATH = r"C:\Users\Mohamed\AppData\Local\Programs\ADI\LTspice\LTspice.exe"
    base_dir = r"C:\Users\Mohamed\Desktop\ML\Redo_project"
    netlist_path = os.path.join(base_dir, "cs_amp.sp")
    raw_path = os.path.join(base_dir, "cs_amp.raw")

    # 2. UPDATE PARAMETERS IN THE NETLIST
    # Read the existing file
    with open(netlist_path, 'r') as f:
        lines = f.readlines()

    # Create new content with updated parameters
    new_lines = []
    for line in lines:
        upper_line = line.upper().strip()
        if upper_line.startswith(".PARAM RD="):
            new_lines.append(f".param Rd={Rd}\n")
        elif upper_line.startswith(".PARAM WN="):
            new_lines.append(f".param Wn={W}\n")
        elif upper_line.startswith(".PARAM VG="):
            new_lines.append(f".param VG={VG}\n")
        elif upper_line.startswith(".PARAM LN="):
            new_lines.append(f".param Ln={L}\n")
        else:
            new_lines.append(line)

    # Write the modified netlist back to disk
    with open(netlist_path, 'w') as f:
        f.writelines(new_lines)

    # 3. RUN SIMULATION
    subprocess.run([LTSPICE_PATH, "-Run", "-b", netlist_path], check=True)

    # 4. DATA EXTRACTION
    if os.path.exists(raw_path):
        LTR = RawRead(raw_path)
        freqs = LTR.get_trace("frequency").get_wave()
        vout_complex = LTR.get_trace("V(out)").get_wave()
        magnitudes = np.abs(vout_complex)

        # Gain calculation
        gain = np.max(magnitudes)
        
        # Bandwidth calculation
        target_mag = gain / np.sqrt(2)
        try:
            idx_bw = np.where(magnitudes <= target_mag)[0][0]
            bw = np.abs(freqs[idx_bw])
        except IndexError:
            bw = 0 # Out of range

        return gain, bw
    return None, None

# --- MAIN EXECUTION ---
# Input your new parameters here
new_L  = 5e-06
new_W  = 0.0002
new_VG = 1.1
new_Rd = 25000.0

gain, bw = run_simulation(new_L, new_W, new_VG, new_Rd)

print("\n--- Simulation Results with New Parameters ---")
print(f"L: {new_L}, W: {new_W}, VG: {new_VG}, Rd: {new_Rd}")
print(f"Gain: {gain:.4f}")
print(f"BW:   {bw/1e6:.2f} MHz")


--- Simulation Results with New Parameters ---
L: 5e-06, W: 0.0002, VG: 1.1, Rd: 25000.0
Gain: 4.7607
BW:   15.85 MHz


#
#

In [2]:
# Data set generation using multpile calls + Randomn selection

import numpy as np
from scipy.io import savemat



# ================= USER SETTINGS =================
N = 10000   # Number of simulations

# Dense grids (for uniform coverage)
L_values  = np.linspace(0.18e-6, 2e-6, 50)     # Channel length
W_values  = np.linspace(20e-6, 300e-6, 50)     # Channel width
VG_values = np.linspace(0.8, 1.5, 60)          # Gate bias
RD_values = np.linspace(5e3, 100e3, 50)        # Drain resistor






# ================= PRE-ALLOCATION =================
X = np.zeros((N, 4))   # [Rd Wn VG Ln]
Y = np.zeros((N, 2))   # [GAIN BW]

valid_count = 0










# ================= DATA GENERATION LOOP =================
for k in range(N):

    # ----- Sample from dense grids (KEY CHANGE) -----
    Ln = np.random.choice(L_values)
    Wn = np.random.choice(W_values)
    VG = np.random.choice(VG_values)
    Rd = np.random.choice(RD_values)

    try:
        # ----- Run LTspice simulation -----
        GAIN, BW = run_simulation(Ln, Wn, VG, Rd)

        # ----- Validate outputs -----
        if (
            GAIN is None or BW is None or
            np.isnan(GAIN) or np.isnan(BW) or
            GAIN <= 0 or BW <= 0
        ):
            continue

        # ----- Store results -----
        X[valid_count, :] = [Rd, Wn, VG, Ln]
        Y[valid_count, :] = [GAIN, BW]
        valid_count += 1

    except Exception:
        # LTspice failure -> skip
        continue

    # ----- Progress display -----
    if (k + 1) % 500 == 0:
        print(f"Completed {k+1} / {N} simulations")








# ================= CLEAN DATA =================
X = X[:valid_count, :]
Y = Y[:valid_count, :]

print(f"Valid simulations: {valid_count} / {N}")



np.savez("cs_dataset.npz", X=X, Y=Y)













Completed 500 / 10000 simulations
Completed 1000 / 10000 simulations
Completed 1500 / 10000 simulations
Completed 2000 / 10000 simulations
Completed 2500 / 10000 simulations
Completed 3000 / 10000 simulations
Completed 3500 / 10000 simulations
Completed 4000 / 10000 simulations
Completed 4500 / 10000 simulations
Completed 5500 / 10000 simulations
Completed 6000 / 10000 simulations
Completed 6500 / 10000 simulations
Completed 7000 / 10000 simulations
Completed 7500 / 10000 simulations
Completed 8000 / 10000 simulations
Completed 9500 / 10000 simulations
Completed 10000 / 10000 simulations
Valid simulations: 8223 / 10000


In [ ]:
# Data set generation using multiple calls + Grid sweep

# ================= USER SETTINGS =================
N_target = 10000

L_values  = np.linspace(1e-6, 5e-6, 10)
W_values  = np.linspace(50e-6, 200e-6, 10)
VG_values = np.linspace(0.8, 1.3, 10)
RD_values = np.linspace(10e3, 50e3, 10)

# ================= STORAGE =================
X = []
Y = []

sim_count = 0
valid_count = 0

# ================= SWEEP =================
for Ln in L_values:
    for Wn in W_values:
        for VG in VG_values:
            for Rd in RD_values:

                if valid_count >= N_target:
                    break

                sim_count += 1

                try:
                    GAIN, BW = run_simulation(Ln, Wn, VG, Rd)

                    if np.isnan(GAIN) or np.isnan(BW) or GAIN <= 0 or BW <= 0:
                        continue

                    X.append([Ln, Wn, VG, Rd])
                    Y.append([GAIN, BW])
                    valid_count += 1

                except Exception:
                    continue

                if sim_count % 500 == 0:
                    print(f"Completed {sim_count} simulations")

# ================= CONVERT =================
X = np.array(X)
Y = np.array(Y)

# ================= SAVE =================
np.savez(
    "cs_dataset_1.npz",
    X=X,
    Y=Y
)

print("\n=========== DONE ===========")
print(f"Total simulations : {sim_count}")
print(f"Valid samples     : {valid_count}")
print(f"X shape           : {X.shape}")
print(f"Y shape           : {Y.shape}")
print("Saved as cs_dataset_1.npz")


In [17]:
 
# test multiple calls using parallel processing

import os
import subprocess
import numpy as np
from PyLTSpice import RawRead

def run_simulation_standalone(L, W, VG, Rd, sim_id):
    # Paths - Change these if you move your files
    LTSPICE_PATH = r"C:\Users\Mohamed\AppData\Local\Programs\ADI\LTspice\LTspice.exe"
    base_dir = r"C:\Users\Mohamed\Desktop\ML\Redo_project"
    
    # Unique filenames to prevent crashes
    netlist_path = os.path.join(base_dir, f"test_sim_{sim_id}.sp")
    raw_path = os.path.join(base_dir, f"test_sim_{sim_id}.raw")
    template_path = os.path.join(base_dir, "cs_amp.sp")

    # 1. Update Parameters
    with open(template_path, 'r') as f:
        lines = f.readlines()

    with open(netlist_path, 'w') as f:
        for line in lines:
            u = line.upper().strip()
            if u.startswith(".PARAM RD="): f.write(f".param Rd={Rd}\n")
            elif u.startswith(".PARAM WN="): f.write(f".param Wn={W}\n")
            elif u.startswith(".PARAM VG="): f.write(f".param VG={VG}\n")
            elif u.startswith(".PARAM LN="): f.write(f".param Ln={L}\n")
            else: f.write(line + "\n")

    # 2. Run LTspice
    # We use "-b" for batch mode so no windows pop up
    subprocess.run([LTSPICE_PATH, "-Run", "-b", netlist_path], check=True)

    # 3. Extract Data
    if os.path.exists(raw_path):
        LTR = RawRead(raw_path)
        freqs = LTR.get_trace("frequency").get_wave()
        vout_complex = LTR.get_trace("V(out)").get_wave()
        magnitudes = np.abs(vout_complex)

        gain = np.max(magnitudes)
        target_mag = gain / np.sqrt(2)
        
        try:
            idx_bw = np.where(magnitudes <= target_mag)[0][0]
            bw = np.abs(freqs[idx_bw])
        except:
            bw = 0

        return gain, bw
    
    return None, None


# new_L  = 5e-06
# new_W  = 0.0002
# new_VG = 1.1
# new_Rd = 25000.0


    # --- TEST DATA ---
# We use L=0.3u instead of 0.18u to avoid the Fatal Error
test_L = 5e-6  
test_W = 0.0002
test_VG = 1.1
test_Rd = 25000

print("Starting 5 Test Simulations...")
print("-" * 30)

for i in range(5):
    # We add a tiny bit of variation to the gate voltage for each test
    current_vg = test_VG + (i * 0.005) 
    
    gain, bw = run_simulation_standalone(test_L, test_W, current_vg, test_Rd, i)
    
    if gain is not None:
        print(f"Test {i}: VG={current_vg:.2f}V | GAIN={gain:.2f} | BW={bw/1e6:.2f}MHz")
    else:
        print(f"Test {i}: FAILED (Check the .log file in your folder)")

print("-" * 30)
print("Testing Complete.")

Starting 5 Test Simulations...
------------------------------
Test 0: VG=1.10V | GAIN=4.76 | BW=15.85MHz
Test 1: VG=1.10V | GAIN=2.85 | BW=24.55MHz
Test 2: VG=1.11V | GAIN=2.13 | BW=31.62MHz
Test 3: VG=1.11V | GAIN=1.72 | BW=37.15MHz
Test 4: VG=1.12V | GAIN=1.46 | BW=43.65MHz
------------------------------
Testing Complete.


In [19]:
# Data set generation using multiple calls + randomn selection + parallel processing but save o/p files problem

import os
import subprocess
import numpy as np
from PyLTSpice import RawRead
from concurrent.futures import ThreadPoolExecutor, as_completed

# ================= 1. THE VERIFIED FUNCTION =================
def run_simulation_parallel(L, W, VG, Rd, sim_id):
    LTSPICE_PATH = r"C:\Users\Mohamed\AppData\Local\Programs\ADI\LTspice\LTspice.exe"
    base_dir = r"C:\Users\Mohamed\Desktop\ML\Redo_project"
    
    netlist_path = os.path.join(base_dir, f"sim_{sim_id}.sp")
    raw_path = os.path.join(base_dir, f"sim_{sim_id}.raw")
    template_path = os.path.join(base_dir, "cs_amp.sp")

    try:
        # Update Parameters
        with open(template_path, 'r') as f:
            lines = f.readlines()

        with open(netlist_path, 'w') as f:
            for line in lines:
                u = line.upper().strip()
                if u.startswith(".PARAM RD="): f.write(f".param Rd={Rd}\n")
                elif u.startswith(".PARAM WN="): f.write(f".param Wn={W}\n")
                elif u.startswith(".PARAM VG="): f.write(f".param VG={VG}\n")
                elif u.startswith(".PARAM LN="): f.write(f".param Ln={L}\n")
                else: f.write(line + "\n")

        # Run LTspice - Using a timeout so it can't hang forever
        subprocess.run([LTSPICE_PATH, "-Run", "-b", netlist_path], 
                       check=True, capture_output=True, timeout=30)

        # Extract Data
        if os.path.exists(raw_path):
            LTR = RawRead(raw_path)
            vout_complex = LTR.get_trace("V(out)").get_wave()
            freqs = LTR.get_trace("frequency").get_wave()
            magnitudes = np.abs(vout_complex)

            gain = np.max(magnitudes)
            target_mag = gain / np.sqrt(2)
            
            try:
                idx_bw = np.where(magnitudes <= target_mag)[0][0]
                bw = np.abs(freqs[idx_bw])
            except:
                bw = 0

            # CLEANUP
            os.remove(netlist_path)
            os.remove(raw_path)
            log_p = netlist_path.replace(".sp", ".log")
            if os.path.exists(log_p): os.remove(log_p)

            return [Rd, W, VG, L], [gain, bw]
    
    except Exception as e:
        # This prevents the "Broken Pool" - we just return None and move on
        return None

# ================= 2. THE MAIN ENGINE =================
if __name__ == '__main__':
    N = 10000
    # Use ThreadPool instead of ProcessPool. 
    # This is MUCH more stable for calling external .exe files like LTspice.
    MAX_SIMULTANEOUS_SIMS = 4 

    L_values  = np.linspace(0.3e-6, 2e-6, 50)
    W_values  = np.linspace(20e-6, 300e-6, 50)
    VG_values = np.linspace(0.8, 1.5, 60)
    RD_values = np.linspace(5e3, 80e3, 50)

    X_data, Y_data = [], []
    
    print(f"Starting Stability-First Engine. Workers: {MAX_SIMULTANEOUS_SIMS}")

    with ThreadPoolExecutor(max_workers=MAX_SIMULTANEOUS_SIMS) as executor:
        # Create all the tasks
        future_to_id = {}
        for i in range(N):
            L = np.random.choice(L_values)
            W = np.random.choice(W_values)
            VG = np.random.choice(VG_values)
            Rd = np.random.choice(RD_values)
            
            f = executor.submit(run_simulation_parallel, L, W, VG, Rd, i)
            future_to_id[f] = i

        count = 0
        for future in as_completed(future_to_id):
            count += 1
            try:
                result = future.result()
                if result:
                    X_data.append(result[0])
                    Y_data.append(result[1])
            except Exception:
                pass # Ignore crashes of individual LTspice instances

            if count % 100 == 0:
                print(f"Done: {count}/{N} | Valid: {len(X_data)}")

    # SAVE
    np.savez("cs_dataset_final.npz", X=np.array(X_data), Y=np.array(Y_data))
    print(f"Finished! Total valid samples: {len(X_data)}")

Starting Stability-First Engine. Workers: 4
Done: 100/10000 | Valid: 100
Done: 200/10000 | Valid: 200
Done: 300/10000 | Valid: 300
Done: 400/10000 | Valid: 400
Done: 500/10000 | Valid: 500
Done: 600/10000 | Valid: 600
Done: 700/10000 | Valid: 700
Done: 800/10000 | Valid: 800
Done: 900/10000 | Valid: 900
Done: 1000/10000 | Valid: 1000
Done: 1100/10000 | Valid: 1100
Done: 1200/10000 | Valid: 1200
Done: 1300/10000 | Valid: 1300
Done: 1400/10000 | Valid: 1400
Done: 1500/10000 | Valid: 1500
Done: 1600/10000 | Valid: 1600
Done: 1700/10000 | Valid: 1700
Done: 1800/10000 | Valid: 1800
Done: 1900/10000 | Valid: 1900
Done: 2000/10000 | Valid: 2000
Done: 2100/10000 | Valid: 2100
Done: 2200/10000 | Valid: 2200
Done: 2300/10000 | Valid: 2300
Done: 2400/10000 | Valid: 2400
Done: 2500/10000 | Valid: 2500
Done: 2600/10000 | Valid: 2600
Done: 2700/10000 | Valid: 2700
Done: 2800/10000 | Valid: 2800
Done: 2900/10000 | Valid: 2900
Done: 3000/10000 | Valid: 3000
Done: 3100/10000 | Valid: 3100
Done: 3200/10

In [40]:
import numpy as np

# Load the dataset
data = np.load("cs_dataset_final.npz")

# Extract the arrays
X = data['X']  # Inputs: [Rd, Wn, VG, Ln]
Y = data['Y']  # Outputs: [GAIN, BW]

print(f"Dataset Loaded Successfully!")
print(f"Total Samples: {X.shape[0]}")
print("-" * 30)
print(f"First 5 Input samples (Rd, Wn, VG, Ln):\n{X[:100]}")
print(f"\nFirst 5 Output samples (Gain, BW):\n{Y[:100]}")

Dataset Loaded Successfully!
Total Samples: 10000
------------------------------
First 5 Input samples (Rd, Wn, VG, Ln):
[[1.26530612e+04 2.25714286e-04 8.35593220e-01 1.06326531e-06]
 [6.16326531e+04 2.94285714e-04 1.47627119e+00 8.55102041e-07]
 [2.03061224e+04 1.51428571e-04 1.22711864e+00 1.30612245e-06]
 [7.69387755e+04 1.85714286e-04 1.39322034e+00 6.12244898e-07]
 [6.16326531e+04 2.20000000e-04 9.18644068e-01 1.68775510e-06]
 [1.57142857e+04 1.62857143e-04 1.20338983e+00 1.72244898e-06]
 [2.33673469e+04 1.62857143e-04 1.50000000e+00 4.73469388e-07]
 [6.53061224e+03 2.54285714e-04 9.18644068e-01 1.23673469e-06]
 [7.08163265e+04 2.88571429e-04 9.66101695e-01 1.09795918e-06]
 [4.47959184e+04 7.71428571e-05 1.12033898e+00 8.55102041e-07]
 [9.59183673e+03 2.08571429e-04 1.12033898e+00 9.24489796e-07]
 [2.33673469e+04 2.60000000e-04 1.50000000e+00 9.59183673e-07]
 [6.53061224e+03 2.42857143e-04 1.22711864e+00 6.81632653e-07]
 [4.78571429e+04 1.22857143e-04 1.06101695e+00 7.51020408e-0

In [36]:
# Data set generation using multiple calls + randomn selection + parallel processing but save o/p files problem
# Just shitier

import os
import subprocess
import numpy as np
from PyLTSpice import RawRead
from concurrent.futures import ThreadPoolExecutor, as_completed

# ================= 1. THE REVERTED FUNCTION =================
def run_simulation_parallel(L, W, VG, Rd, sim_id):
    LTSPICE_PATH = r"C:\Users\Mohamed\AppData\Local\Programs\ADI\LTspice\LTspice.exe"
    # Everything stays in the main project folder so it can see your models!
    base_dir = r"C:\Users\Mohamed\Desktop\ML\Redo_project"
    
    netlist_path = os.path.join(base_dir, f"sim_{sim_id}.sp")
    raw_path = os.path.join(base_dir, f"sim_{sim_id}.raw")
    template_path = os.path.join(base_dir, "cs_amp.sp")

    try:
        # Step A: Update Parameters
        with open(template_path, 'r') as f:
            lines = f.readlines()

        with open(netlist_path, 'w') as f:
            for line in lines:
                u = line.upper().strip()
                if u.startswith(".PARAM RD="): f.write(f".param Rd={Rd}\n")
                elif u.startswith(".PARAM WN="): f.write(f".param Wn={W}\n")
                elif u.startswith(".PARAM VG="): f.write(f".param VG={VG}\n")
                elif u.startswith(".PARAM LN="): f.write(f".param Ln={L}\n")
                else: f.write(line + "\n")

        # Step B: Run LTspice (Running in base_dir ensures models are found)
        subprocess.run([LTSPICE_PATH, "-Run", "-b", netlist_path], 
                       cwd=base_dir, check=True, capture_output=True, timeout=30)

        # Step C: Data Extraction & Filtering
        if os.path.exists(raw_path):
            LTR = RawRead(raw_path)
            vout_complex = LTR.get_trace("V(out)").get_wave()
            freqs = LTR.get_trace("frequency").get_wave()
            magnitudes = np.abs(vout_complex)

            gain = np.max(magnitudes)
            
            # --- FILTER: Keep only working amplifiers ---
            if gain > 0 and not np.isnan(gain):
                target_mag = gain / np.sqrt(2)
                try:
                    idx_bw = np.where(magnitudes <= target_mag)[0][0]
                    bw = np.abs(freqs[idx_bw])
                except:
                    bw = 0
                
                # --- CLEANUP IMMEDIATELY ---
                os.remove(netlist_path)
                os.remove(raw_path)
                log_f = netlist_path.replace(".sp", ".log")
                if os.path.exists(log_f): os.remove(log_f)

                return [Rd, W, VG, L], [gain, bw]
        
        # Cleanup even if it failed
        if os.path.exists(netlist_path): os.remove(netlist_path)
        return None
    
    except Exception:
        return None

# ================= 2. THE MAIN EXECUTION =================
if __name__ == '__main__':
    N = 10000
    MAX_WORKERS = 4 

    # Your verified working ranges
    L_values  = np.linspace(0.3e-6, 5e-6, 50)
    W_values  = np.linspace(20e-6, 300e-6, 50)
    VG_values = np.linspace(0.8, 1.5, 60)
    RD_values = np.linspace(5e3, 80e3, 50)

    X_list, Y_list = [], []
    print(f"Starting Parallel Engine. Attempts: {N}")

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = []
        for i in range(N):
            L, W = np.random.choice(L_values), np.random.choice(W_values)
            VG, Rd = np.random.choice(VG_values), np.random.choice(RD_values)
            futures.append(executor.submit(run_simulation_parallel, L, W, VG, Rd, i))

        for count, future in enumerate(as_completed(futures), 1):
            res = future.result()
            if res:
                X_list.append(res[0])
                Y_list.append(res[1])

            if count % 200 == 0:
                print(f"Progress: {count}/{N} | Valid: {len(X_list)}")

    # Save
    if len(X_list) > 0:
        np.savez("cs_dataset_final_repeat.npz", X=np.array(X_list), Y=np.array(Y_list))
        print(f"Finished! Saved {len(X_list)} valid samples.")
    else:
        print("Still 0 samples. Check if your models are in Redo_project folder.")

Starting Parallel Engine. Attempts: 10
Finished! Saved 10 valid samples.


In [38]:
import numpy as np

# Load the dataset
data = np.load("cs_dataset_final_repeat.npz")

# Extract the arrays
X = data['X']  # Inputs: [Rd, Wn, VG, Ln]
Y = data['Y']  # Outputs: [GAIN, BW]

print(f"Dataset Loaded Successfully!")
print(f"Total Samples: {X.shape[0]}")
print("-" * 30)
print(f"First 5 Input samples (Rd, Wn, VG, Ln):\n{X[:100]}")
print(f"\nFirst 5 Output samples (Gain, BW):\n{Y[:100]}")

Dataset Loaded Successfully!
Total Samples: 10
------------------------------
First 5 Input samples (Rd, Wn, VG, Ln):
[[5.24489796e+04 9.42857143e-05 8.59322034e-01 1.93061224e-06]
 [2.33673469e+04 2.57142857e-05 1.03728814e+00 1.35510204e-06]
 [2.33673469e+04 1.28571429e-04 9.54237288e-01 4.13673469e-06]
 [3.40816327e+04 8.28571429e-05 1.21525424e+00 3.95918367e-07]
 [9.59183673e+03 1.57142857e-04 1.41694915e+00 7.79591837e-07]
 [4.78571429e+04 2.20000000e-04 1.12033898e+00 2.60204082e-06]
 [1.72448980e+04 1.34285714e-04 1.06101695e+00 4.13673469e-06]
 [1.11224490e+04 1.05714286e-04 1.42881356e+00 4.61632653e-06]
 [7.38775510e+04 3.00000000e-04 1.33389831e+00 1.73877551e-06]
 [6.16326531e+04 2.57142857e-05 8.71186441e-01 2.98571429e-06]]

First 5 Output samples (Gain, BW):
[[2.41006056e+01 3.09029543e+06]
 [1.11759345e+01 7.76247117e+06]
 [1.24467951e+01 6.76082975e+06]
 [9.68054133e-03 0.00000000e+00]
 [4.85790662e-02 0.00000000e+00]
 [2.71614860e-01 1.00000000e+03]
 [1.50593121e+01 

In [42]:
# Data set generation using multiple calls + randomn selection + parallel processing + removes save o/p files (cleaner)


import os
import subprocess
import numpy as np
from PyLTSpice import RawRead
from concurrent.futures import ThreadPoolExecutor, as_completed

# ================= 1. THE REVERTED & CLEANING FUNCTION =================
def run_simulation_parallel(L, W, VG, Rd, sim_id):
    LTSPICE_PATH = r"C:\Users\Mohamed\AppData\Local\Programs\ADI\LTspice\LTspice.exe"
    base_dir = r"C:\Users\Mohamed\Desktop\ML\Redo_project"
    
    # Unique filenames to avoid collisions
    netlist_path = os.path.join(base_dir, f"sim_{sim_id}.sp")
    raw_path = os.path.join(base_dir, f"sim_{sim_id}.raw")
    template_path = os.path.join(base_dir, "cs_amp.sp")

    try:
        # Step A: Update Parameters in unique file
        with open(template_path, 'r') as f:
            lines = f.readlines()

        with open(netlist_path, 'w') as f:
            for line in lines:
                u = line.upper().strip()
                if u.startswith(".PARAM RD="): f.write(f".param Rd={Rd}\n")
                elif u.startswith(".PARAM WN="): f.write(f".param Wn={W}\n")
                elif u.startswith(".PARAM VG="): f.write(f".param VG={VG}\n")
                elif u.startswith(".PARAM LN="): f.write(f".param Ln={L}\n")
                else: f.write(line + "\n")

        # Step B: Run LTspice (cwd ensures it finds models in the same folder)
        subprocess.run([LTSPICE_PATH, "-Run", "-b", netlist_path], 
                       cwd=base_dir, check=True, capture_output=True, timeout=30)

        # Step C: Data Extraction
        result = None
        if os.path.exists(raw_path):
            LTR = RawRead(raw_path)
            vout_complex = LTR.get_trace("V(out)").get_wave()
            freqs = LTR.get_trace("frequency").get_wave()
            magnitudes = np.abs(vout_complex)

            gain = np.max(magnitudes)
            
            # --- FILTER: Keep only functional results ---
            if gain > 0 and not np.isnan(gain):
                target_mag = gain / np.sqrt(2)
                try:
                    idx_bw = np.where(magnitudes <= target_mag)[0][0]
                    bw = np.abs(freqs[idx_bw])
                except:
                    bw = 0
                
                result = ([Rd, W, VG, L], [gain, bw])

        # --- STEP D: THE CLEANUP CREW ---
        # This removes the .sp, .raw, and .log files for THIS specific simulation
        for ext in [".sp", ".raw", ".log", ".op.raw"]:
            file_to_del = netlist_path.replace(".sp", ext)
            if os.path.exists(file_to_del):
                os.remove(file_to_del)

        return result
    
    except Exception:
        # Cleanup if it crashed during run
        if os.path.exists(netlist_path): os.remove(netlist_path)
        return None

# ================= 2. THE MAIN ENGINE =================
if __name__ == '__main__':
    N = 100 
    MAX_WORKERS = 4 # Keeps Windows stable

    # Ranges from your working test
    L_values  = np.linspace(0.3e-6, 5e-6, 50)
    W_values  = np.linspace(20e-6, 300e-6, 50)
    VG_values = np.linspace(0.8, 1.5, 60)
    RD_values = np.linspace(5e3, 80e3, 50)

    X_list, Y_list = [], []
    print(f"Engine Running. Folder: {os.getcwd()}")

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = []
        for i in range(N):
            L = np.random.choice(L_values)
            W = np.random.choice(W_values)
            VG = np.random.choice(VG_values)
            Rd = np.random.choice(RD_values)
            futures.append(executor.submit(run_simulation_parallel, L, W, VG, Rd, i))

        for count, future in enumerate(as_completed(futures), 1):
            res = future.result()
            if res:
                X_list.append(res[0])
                Y_list.append(res[1])

            if count % 200 == 0:
                print(f"Total: {count}/{N} | Valid: {len(X_list)}")

    # ================= 3. FINAL SAVE =================
    if len(X_list) > 0:
        np.savez("cs_dataset_final_repeated.npz", X=np.array(X_list), Y=np.array(Y_list))
        print(f"DONE! Saved {len(X_list)} valid samples. No junk files left in folder.")
    else:
        print("Still 0 samples. Verify your model paths in cs_amp.sp.")

Engine Running. Folder: C:\Users\Mohamed\Desktop\ML\Redo_project
DONE! Saved 100 valid samples. No junk files left in folder.


In [43]:
import numpy as np

# Load the dataset
data = np.load("cs_dataset_final_repeated.npz")

# Extract the arrays
X = data['X']  # Inputs: [Rd, Wn, VG, Ln]
Y = data['Y']  # Outputs: [GAIN, BW]

print(f"Dataset Loaded Successfully!")
print(f"Total Samples: {X.shape[0]}")
print("-" * 30)
print(f"First 5 Input samples (Rd, Wn, VG, Ln):\n{X[:100]}")
print(f"\nFirst 5 Output samples (Gain, BW):\n{Y[:100]}")

Dataset Loaded Successfully!
Total Samples: 100
------------------------------
First 5 Input samples (Rd, Wn, VG, Ln):
[[2.94897959e+04 1.11428571e-04 1.44067797e+00 7.79591837e-07]
 [1.72448980e+04 1.45714286e-04 1.47627119e+00 4.23265306e-06]
 [6.92857143e+04 1.22857143e-04 1.42881356e+00 4.04081633e-06]
 [3.71428571e+04 2.54285714e-04 1.44067797e+00 1.83469388e-06]
 [5.39795918e+04 1.74285714e-04 1.20338983e+00 4.80816327e-06]
 [6.92857143e+04 2.08571429e-04 1.19152542e+00 1.64285714e-06]
 [5.70408163e+04 6.00000000e-05 1.12033898e+00 4.04081633e-06]
 [6.31632653e+04 2.31428571e-04 1.15593220e+00 3.17755102e-06]
 [7.69387755e+04 2.48571429e-04 8.47457627e-01 3.94489796e-06]
 [5.70408163e+04 2.31428571e-04 1.10847458e+00 3.56122449e-06]
 [5.24489796e+04 1.11428571e-04 1.15593220e+00 4.61632653e-06]
 [6.16326531e+04 8.85714286e-05 1.07288136e+00 2.02653061e-06]
 [4.47959184e+04 2.00000000e-05 8.71186441e-01 1.83469388e-06]
 [1.11224490e+04 1.00000000e-04 1.19152542e+00 4.13673469e-06]

In [44]:
# Data set generation using multiple calls + randomn selection + parallel processing + removes save o/p files (cleaner)
# + removes useless data sets 

import os
import subprocess
import numpy as np
from PyLTSpice import RawRead
from concurrent.futures import ThreadPoolExecutor, as_completed

# ================= 1. THE FILTERED SIMULATION FUNCTION =================
def run_simulation_parallel(L, W, VG, Rd, sim_id):
    LTSPICE_PATH = r"C:\Users\Mohamed\AppData\Local\Programs\ADI\LTspice\LTspice.exe"
    base_dir = r"C:\Users\Mohamed\Desktop\ML\Redo_project"
    
    netlist_path = os.path.join(base_dir, f"sim_{sim_id}.sp")
    raw_path = os.path.join(base_dir, f"sim_{sim_id}.raw")
    template_path = os.path.join(base_dir, "cs_amp.sp")

    try:
        # Step A: Update Parameters
        with open(template_path, 'r') as f:
            lines = f.readlines()

        with open(netlist_path, 'w') as f:
            for line in lines:
                u = line.upper().strip()
                if u.startswith(".PARAM RD="): f.write(f".param Rd={Rd}\n")
                elif u.startswith(".PARAM WN="): f.write(f".param Wn={W}\n")
                elif u.startswith(".PARAM VG="): f.write(f".param VG={VG}\n")
                elif u.startswith(".PARAM LN="): f.write(f".param Ln={L}\n")
                else: f.write(line + "\n")

        # Step B: Run LTspice
        subprocess.run([LTSPICE_PATH, "-Run", "-b", netlist_path], 
                       cwd=base_dir, check=True, capture_output=True, timeout=30)

        # Step C: Data Extraction and Filtering
        result = None
        if os.path.exists(raw_path):
            LTR = RawRead(raw_path)
            vout_complex = LTR.get_trace("V(out)").get_wave()
            freqs = LTR.get_trace("frequency").get_wave()
            magnitudes = np.abs(vout_complex)

            gain = np.max(magnitudes)
            
            # Find BW
            target_mag = gain / np.sqrt(2)
            try:
                idx_bw = np.where(magnitudes <= target_mag)[0][0]
                bw = np.abs(freqs[idx_bw])
            except:
                bw = 0
            
            # --- CRITICAL FILTERING STEP ---
            # Only return data if the circuit is actually working
            # You can set gain > 1.0 if you only want 'amplifying' results
            if gain > 0 and bw > 0:
                result = ([Rd, W, VG, L], [gain, bw])

        # Step D: Cleanup immediately
        for ext in [".sp", ".raw", ".log", ".op.raw"]:
            file_to_del = netlist_path.replace(".sp", ext)
            if os.path.exists(file_to_del):
                os.remove(file_to_del)

        return result
    
    except Exception:
        if os.path.exists(netlist_path): os.remove(netlist_path)
        return None



        

# ================= 2. THE MAIN ENGINE =================
if __name__ == '__main__':
    N = 100 
    MAX_WORKERS = 4 

    L_values  = np.linspace(0.3e-6, 5e-6, 50)
    W_values  = np.linspace(20e-6, 300e-6, 50)
    VG_values = np.linspace(0.8, 1.5, 60)
    RD_values = np.linspace(5e3, 80e3, 50)

    X_list, Y_list = [], []
    print(f"Starting Filtered Engine. Attempts: {N}")

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = []
        for i in range(N):
            L, W = np.random.choice(L_values), np.random.choice(W_values)
            VG, Rd = np.random.choice(VG_values), np.random.choice(RD_values)
            futures.append(executor.submit(run_simulation_parallel, L, W, VG, Rd, i))

        for count, future in enumerate(as_completed(futures), 1):
            res = future.result()
            
            # Only append if validation passed (both Gain and BW > 0)
            if res is not None:
                X_list.append(res[0])
                Y_list.append(res[1])

            if count % 200 == 0:
                print(f"Attempts: {count}/{N} | Valid (Working) Samples: {len(X_list)}")

    # ================= 3. SAVE =================
    if len(X_list) > 0:
        X_final = np.array(X_list)
        Y_final = np.array(Y_list)
        np.savez("cs_dataset_final_repeatedly.npz", X=X_final, Y=Y_final)
        print("-" * 30)
        print(f"DONE! Saved {len(X_final)} high-quality samples.")
    else:
        print("No valid samples found. Check biasing conditions.")

Starting Filtered Engine. Attempts: 100
------------------------------
DONE! Saved 83 high-quality samples.


In [45]:
import numpy as np

# Load the dataset
data = np.load("cs_dataset_final_repeatedly.npz")

# Extract the arrays
X = data['X']  # Inputs: [Rd, Wn, VG, Ln]
Y = data['Y']  # Outputs: [GAIN, BW]

print(f"Dataset Loaded Successfully!")
print(f"Total Samples: {X.shape[0]}")
print("-" * 30)
print(f"First 5 Input samples (Rd, Wn, VG, Ln):\n{X[:100]}")
print(f"\nFirst 5 Output samples (Gain, BW):\n{Y[:100]}")

Dataset Loaded Successfully!
Total Samples: 83
------------------------------
First 5 Input samples (Rd, Wn, VG, Ln):
[[6.01020408e+04 2.25714286e-04 1.32203390e+00 5.00000000e-06]
 [3.25510204e+04 2.88571429e-04 1.39322034e+00 9.71428571e-07]
 [3.56122449e+04 2.88571429e-04 1.33389831e+00 2.60204082e-06]
 [6.31632653e+04 3.71428571e-05 1.31016949e+00 4.42448980e-06]
 [1.57142857e+04 2.57142857e-05 1.25084746e+00 3.08163265e-06]
 [8.06122449e+03 1.97142857e-04 1.14406780e+00 4.71224490e-06]
 [5.24489796e+04 1.40000000e-04 1.29830508e+00 4.90408163e-06]
 [3.56122449e+04 1.05714286e-04 9.18644068e-01 2.60204082e-06]
 [1.11224490e+04 1.62857143e-04 1.27457627e+00 3.27346939e-06]
 [6.31632653e+04 1.05714286e-04 1.02542373e+00 2.88979592e-06]
 [7.84693878e+04 2.02857143e-04 1.41694915e+00 4.61632653e-06]
 [4.47959184e+04 2.94285714e-04 1.21525424e+00 3.65714286e-06]
 [1.11224490e+04 8.28571429e-05 1.26271186e+00 1.25918367e-06]
 [6.62244898e+04 2.42857143e-04 1.14406780e+00 4.13673469e-06]


In [2]:
# Data set generation using multiple calls + randomn selection + parallel processing + removes save o/p files (cleaner)
# + removes useless data sets 

import os
import subprocess
import numpy as np
from PyLTSpice import RawRead
from concurrent.futures import ThreadPoolExecutor, as_completed

# ================= 1. THE FILTERED SIMULATION FUNCTION =================
def run_simulation_parallel(L, W, VG, Rd, sim_id):
    LTSPICE_PATH = r"C:\Users\Mohamed\AppData\Local\Programs\ADI\LTspice\LTspice.exe"
    base_dir = r"C:\Users\Mohamed\Desktop\ML\Redo_project"
    
    netlist_path = os.path.join(base_dir, f"sim_{sim_id}.sp")
    raw_path = os.path.join(base_dir, f"sim_{sim_id}.raw")
    template_path = os.path.join(base_dir, "cs_amp.sp")

    try:
        # Step A: Update Parameters
        with open(template_path, 'r') as f:
            lines = f.readlines()

        with open(netlist_path, 'w') as f:
            for line in lines:
                u = line.upper().strip()
                if u.startswith(".PARAM RD="): f.write(f".param Rd={Rd}\n")
                elif u.startswith(".PARAM WN="): f.write(f".param Wn={W}\n")
                elif u.startswith(".PARAM VG="): f.write(f".param VG={VG}\n")
                elif u.startswith(".PARAM LN="): f.write(f".param Ln={L}\n")
                else: f.write(line + "\n")

        # Step B: Run LTspice
        subprocess.run([LTSPICE_PATH, "-Run", "-b", netlist_path], 
                       cwd=base_dir, check=True, capture_output=True, timeout=30)

        # Step C: Data Extraction and Filtering
        result = None
        if os.path.exists(raw_path):
            LTR = RawRead(raw_path)
            vout_complex = LTR.get_trace("V(out)").get_wave()
            freqs = LTR.get_trace("frequency").get_wave()
            magnitudes = np.abs(vout_complex)

            gain = np.max(magnitudes)
            
            # Find BW
            target_mag = gain / np.sqrt(2)
            try:
                idx_bw = np.where(magnitudes <= target_mag)[0][0]
                bw = np.abs(freqs[idx_bw])
            except:
                bw = 0
            
            # --- CRITICAL FILTERING STEP ---
            # Only return data if the circuit is actually working
            # You can set gain > 1.0 if you only want 'amplifying' results
            if gain > 0 and bw > 0:
                result = ([Rd, W, VG, L], [gain, bw])

        # Step D: Cleanup immediately
        for ext in [".sp", ".raw", ".log", ".op.raw"]:
            file_to_del = netlist_path.replace(".sp", ext)
            if os.path.exists(file_to_del):
                os.remove(file_to_del)

        return result
    
    except Exception:
        if os.path.exists(netlist_path): os.remove(netlist_path)
        return None



        

# ================= 2. THE MAIN ENGINE =================
if __name__ == '__main__':
    N = 10000 
    MAX_WORKERS = 4 

    L_values  = np.linspace(0.3e-6, 5e-6, 50)
    W_values  = np.linspace(20e-6, 300e-6, 50)
    VG_values = np.linspace(0.8, 1.5, 60)
    RD_values = np.linspace(5e3, 80e3, 50)

    X_list, Y_list = [], []
    print(f"Starting Filtered Engine. Attempts: {N}")

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = []
        for i in range(N):
            L, W = np.random.choice(L_values), np.random.choice(W_values)
            VG, Rd = np.random.choice(VG_values), np.random.choice(RD_values)
            futures.append(executor.submit(run_simulation_parallel, L, W, VG, Rd, i))

        for count, future in enumerate(as_completed(futures), 1):
            res = future.result()
            
            # Only append if validation passed (both Gain and BW > 0)
            if res is not None:
                X_list.append(res[0])
                Y_list.append(res[1])

            if count % 1000 == 0:
                print(f"Attempts: {count}/{N} | Valid (Working) Samples: {len(X_list)}")

    # ================= 3. SAVE =================
    if len(X_list) > 0:
        X_final = np.array(X_list)
        Y_final = np.array(Y_list)
        np.savez("cs_dataset_final_justgo.npz", X=X_final, Y=Y_final)
        print("-" * 30)
        print(f"DONE! Saved {len(X_final)} high-quality samples.")
    else:
        print("No valid samples found. Check biasing conditions.")

Starting Filtered Engine. Attempts: 10000
Attempts: 1000/10000 | Valid (Working) Samples: 879
Attempts: 2000/10000 | Valid (Working) Samples: 1747
Attempts: 3000/10000 | Valid (Working) Samples: 2615
Attempts: 4000/10000 | Valid (Working) Samples: 3486
Attempts: 5000/10000 | Valid (Working) Samples: 4355
Attempts: 6000/10000 | Valid (Working) Samples: 5220
Attempts: 7000/10000 | Valid (Working) Samples: 6091
Attempts: 8000/10000 | Valid (Working) Samples: 6961
Attempts: 9000/10000 | Valid (Working) Samples: 7829
Attempts: 10000/10000 | Valid (Working) Samples: 8716
------------------------------
DONE! Saved 8716 high-quality samples.


In [3]:
import numpy as np

# Load the dataset
data = np.load("cs_dataset_final_justgo.npz")

# Extract the arrays
X = data['X']  # Inputs: [Rd, Wn, VG, Ln]
Y = data['Y']  # Outputs: [GAIN, BW]

print(f"Dataset Loaded Successfully!")
print(f"Total Samples: {X.shape[0]}")
print("-" * 30)
print(f"First 5 Input samples (Rd, Wn, VG, Ln):\n{X[:100]}")
print(f"\nFirst 5 Output samples (Gain, BW):\n{Y[:100]}")

Dataset Loaded Successfully!
Total Samples: 8716
------------------------------
First 5 Input samples (Rd, Wn, VG, Ln):
[[5.70408163e+04 9.42857143e-05 1.48813559e+00 1.06734694e-06]
 [6.77551020e+04 2.60000000e-04 1.23898305e+00 2.31428571e-06]
 [6.16326531e+04 2.20000000e-04 8.94915254e-01 4.80816327e-06]
 [7.23469388e+04 1.22857143e-04 9.54237288e-01 1.45102041e-06]
 [2.79591837e+04 1.34285714e-04 8.00000000e-01 7.79591837e-07]
 [7.84693878e+04 2.08571429e-04 1.35762712e+00 3.56122449e-06]
 [7.84693878e+04 1.51428571e-04 9.30508475e-01 4.71224490e-06]
 [2.33673469e+04 2.77142857e-04 9.30508475e-01 1.93061224e-06]
 [3.10204082e+04 2.25714286e-04 1.12033898e+00 1.64285714e-06]
 [4.17346939e+04 2.42857143e-04 1.39322034e+00 4.32857143e-06]
 [3.10204082e+04 2.65714286e-04 1.17966102e+00 3.94489796e-06]
 [4.47959184e+04 2.37142857e-04 1.38135593e+00 2.02653061e-06]
 [5.00000000e+03 2.77142857e-04 1.26271186e+00 2.31428571e-06]
 [5.39795918e+04 2.31428571e-04 1.12033898e+00 3.36938776e-06

In [4]:
import numpy as np
import scipy.io as sio

data = sio.loadmat("cs_datasets.mat")

X = data["X"]
Y = data["Y"]

np.savez("cs_dataset_from_mat.npz", X=X, Y=Y)


data = np.load("cs_dataset_from_mat.npz")
X = data["X"]
Y = data["Y"]


X.shape[1] == 4
Y.shape[1] == 2


True